# 05 — VLM Supervised Fine-Tuning

Turn labelled image sequences into supervised examples and understand masking, batching, and loss.

> **Course principle:** every notebook separates **general-purpose knowledge** from **Manga Task application**.

## Training-example design

A clean training example contains the same kind of input you will use at inference plus the exact target format you want the model to emit.

For this task, sequence-level examples are attractive because speaker labels must remain consistent across all three pages.

In [ ]:
example = {
    "sequence_id": "seq_demo",
    "images": ["page1.jpg", "page2.jpg", "page3.jpg"],
    "target": {
        "sequence_id": "seq_demo",
        "pages": [
            [{"speaker": "char1", "text": "Where are you going?"}],
            [{"speaker": "char2", "text": "Home."}],
            []
        ]
    }
}
example

## Loss masking

For instruction tuning, you often want loss on the assistant target, not on user prompt tokens. Otherwise the model spends capacity learning to reproduce the prompt.

Conceptually:

`[image tokens][instruction tokens][assistant target tokens]`

Labels for prompt positions can be set to `-100` so cross-entropy ignores them.

In [ ]:
import torch

input_ids = torch.tensor([[10, 11, 12, 13, 14, 15]])
assistant_starts_at = 3

labels = input_ids.clone()
labels[:, :assistant_starts_at] = -100
print("input_ids:", input_ids)
print("labels:   ", labels)

## Grouped train/validation split

Never split individual pages from the same sequence across train and validation.

In [ ]:
from sklearn.model_selection import train_test_split

sequence_ids = [f"seq_{i:03d}" for i in range(80)]
train_ids, val_ids = train_test_split(
    sequence_ids, test_size=0.2, random_state=42
)

print(len(train_ids), len(val_ids))
print("Leakage:", bool(set(train_ids) & set(val_ids)))

## Training-loop skeleton

The exact forward signature varies by model family, but the optimization logic is ordinary PyTorch.

In [ ]:
# optimizer = torch.optim.AdamW(
#     (p for p in model.parameters() if p.requires_grad),
#     lr=2e-4
# )

# model.train()
# for batch in train_loader:
#     outputs = model(**batch)
#     loss = outputs.loss
#     optimizer.zero_grad(set_to_none=True)
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#     optimizer.step()